# 03 Incremental Pipeline (Target-Aware Incremental Pipeline Template)

This maintained template organizes work by **target flow**. FabricOps resolves every incremental source from the last Source Observation successfully committed for that exact source → target relationship. Full and incremental sources can be mixed in one flow.

Each successful `pipeline_write()` is an independent publication boundary. In a multi-target notebook, one target can commit while another fails; FabricOps does not provide cross-target atomicity.

## Tested with FabricOps

This template has not yet been manually validated in Microsoft Fabric. Do not add a release to a Fabric-tested table until that validation has actually run.

# 0. Environment

Run the shared Fabric configuration and import the public APIs used by both target flows.

In [ ]:
%run 00_env_config

In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    check_dq,
    check_freshness,
    check_guardrail_coverage,
    check_schema,
    check_sensitive_data,
    check_source_drift,
    pipeline_read,
    pipeline_write,
    profile_table,
    resolve_table_id,
    widget_select_data_contract,
)

# 1. Data Contract

Select the Data Contracts to test with this pipeline. Production automatically uses activated Data Contracts. Read mode remains explicit and separate from each target's governed write strategy.

In [ ]:
CONTRACTS = widget_select_data_contract(spark_session=spark)

# 2. TARGET FLOW 1 — Curated Orders

Define the target first, then resolve each source relative to it. On the first run, the missing committed baseline produces a deterministic complete-source bootstrap. Later runs return only watermark or changed-partition scope.

In [ ]:
# 2.1 Target definition — identity must exist before an incremental read.
TARGET_1_STORE = "Silver"
TARGET_1_SCHEMA = "demo"
TARGET_1_TABLE = "curated_orders"
TARGET_1_LOAD_STRATEGY = "append"
target_1_table_id = resolve_table_id(
    store=TARGET_1_STORE,
    schema=TARGET_1_SCHEMA,
    table_name=TARGET_1_TABLE,
)

In [ ]:
# 2.2 Read Orders incrementally for Target 1.
# FabricOps resolves the committed source → target baseline; do not query metadata manually.
orders_1 = pipeline_read(
    store="Bronze",
    schema="demo",
    table_name="orders",
    read_mode="incremental",
    target_table_id=target_1_table_id,
    spark_session=spark,
)
orders_1_df = orders_1["dataframe"]

# Incremental batches are partial and must not replace the canonical complete-source profile.
# Freshness and Source Drift use the complete physical Source Observation captured by pipeline_read().
check_freshness(orders_1["table_id"])
check_schema(orders_1_df, table_id=orders_1["table_id"])
orders_1_dq = check_dq(orders_1_df, table_id=orders_1["table_id"])
check_source_drift(
    orders_1["table_id"],
    target_table_id=target_1_table_id,
    raise_on_failure=True,
)

In [ ]:
# 2.3 Read Products in full for the same target flow.
products_1 = pipeline_read(
    store="Bronze",
    schema="demo",
    table_name="products",
    read_mode="full",
    target_table_id=target_1_table_id,
    spark_session=spark,
)
products_1_df = products_1["dataframe"]
check_freshness(products_1["table_id"])
check_schema(products_1_df, table_id=products_1["table_id"])
products_1_dq = check_dq(products_1_df, table_id=products_1["table_id"])
# A full read remains eligible for canonical complete-source profiling.
products_1_profile = profile_table(store="Bronze", schema="demo", table_name="products")

In [ ]:
# 2.4–2.7 Transform, check, write, then profile the complete persisted target.
# No-new-data is an explicit safe skip: no physical write and no progress commit occurs.
if orders_1["should_process"]:
    target_1_df = (
        orders_1_dq.get("dataframe", orders_1_df)
        .join(products_1_dq.get("dataframe", products_1_df), on="product_id", how="left")
        .withColumn("processed_at", F.current_timestamp())
    )
    check_schema(target_1_df, table_id=target_1_table_id)
    sensitive_1 = check_sensitive_data(target_1_df, table_id=target_1_table_id)
    target_1_dq = check_dq(sensitive_1["dataframe"], table_id=target_1_table_id)
    check_guardrail_coverage(target_1_table_id)
    target_1_write = pipeline_write(
        target_1_dq.get("dataframe", sensitive_1["dataframe"]),
        store=TARGET_1_STORE,
        schema=TARGET_1_SCHEMA,
        table_name=TARGET_1_TABLE,
        load_strategy=TARGET_1_LOAD_STRATEGY,
        source_table_ids=[orders_1["table_id"], products_1["table_id"]],
        spark_session=spark,
    )
    # Read-back profiling preserves the canonical complete-table profile meaning.
    target_1_profile = profile_table(table_id=target_1_write["table_id"])
else:
    print("Target 1 → no unconsumed Orders data; publication and progress commit skipped.")

# 3. TARGET FLOW 2 — Customer Orders

This cloneable flow reads the same Orders source relative to a different target. Its committed progress is isolated from Target 1.

In [ ]:
# 3.1 Target definition.
TARGET_2_STORE = "Gold"
TARGET_2_SCHEMA = "demo"
TARGET_2_TABLE = "customer_orders"
TARGET_2_LOAD_STRATEGY = "scd1"
TARGET_2_LOAD_PARAMETERS = {"key_columns": ["order_id"]}
target_2_table_id = resolve_table_id(
    store=TARGET_2_STORE,
    schema=TARGET_2_SCHEMA,
    table_name=TARGET_2_TABLE,
)

In [ ]:
# 3.2 Orders incremental relative to Target 2; 3.3 Customers full.
orders_2 = pipeline_read(
    store="Bronze",
    schema="demo",
    table_name="orders",
    read_mode="incremental",
    target_table_id=target_2_table_id,
    spark_session=spark,
)
customers_2 = pipeline_read(
    store="Bronze",
    schema="demo",
    table_name="customers",
    read_mode="full",
    target_table_id=target_2_table_id,
    spark_session=spark,
)
orders_2_df = orders_2["dataframe"]
customers_2_df = customers_2["dataframe"]
check_freshness(orders_2["table_id"])
check_schema(orders_2_df, table_id=orders_2["table_id"])
orders_2_dq = check_dq(orders_2_df, table_id=orders_2["table_id"])
check_source_drift(
    orders_2["table_id"],
    target_table_id=target_2_table_id,
    raise_on_failure=True,
)
check_freshness(customers_2["table_id"])
check_schema(customers_2_df, table_id=customers_2["table_id"])
customers_2_dq = check_dq(customers_2_df, table_id=customers_2["table_id"])
customers_2_profile = profile_table(store="Bronze", schema="demo", table_name="customers")

In [ ]:
# 3.4–3.7 Transform, check, independently publish, and profile Target 2.
if orders_2["should_process"]:
    target_2_df = orders_2_dq.get("dataframe", orders_2_df).join(
        customers_2_dq.get("dataframe", customers_2_df),
        on="customer_id",
        how="left",
    )
    check_schema(target_2_df, table_id=target_2_table_id)
    sensitive_2 = check_sensitive_data(target_2_df, table_id=target_2_table_id)
    target_2_dq = check_dq(sensitive_2["dataframe"], table_id=target_2_table_id)
    check_guardrail_coverage(target_2_table_id)
    target_2_write = pipeline_write(
        target_2_dq.get("dataframe", sensitive_2["dataframe"]),
        store=TARGET_2_STORE,
        schema=TARGET_2_SCHEMA,
        table_name=TARGET_2_TABLE,
        load_strategy=TARGET_2_LOAD_STRATEGY,
        load_strategy_parameters=TARGET_2_LOAD_PARAMETERS,
        source_table_ids=[orders_2["table_id"], customers_2["table_id"]],
        spark_session=spark,
    )
    target_2_profile = profile_table(table_id=target_2_write["table_id"])
else:
    print("Target 2 → no unconsumed Orders data; publication and progress commit skipped.")

## Governed write strategies

Use the target strategy required by the selected or active Data Contract: append for new rows, SCD1 for current-state changes, SCD2 for history, or partition-scoped overwrite for changed partitions. Read mode does not infer write strategy. FabricOps rejects an incremental partial source published through destructive whole-table overwrite.